# Comparing SNN energy with ESN neurons

This notebook shows how to use the built-in energy metadata of the evolved neurons from this repository to compare network energy.

*Disclaimer: The provided values are not realistic absolute energy measurements. They were obtained using FPGA emulation and intentionally ignore parts of a comprehensive hardware energy model. Use them to compare networks built with the default neuron parameters shipped with this repository under the same workload, e.g., to select the most efficient solution. If neuron parameters are changed, the built-in energy metadata no longer applies unless it is recomputed for that configuration. The values are reported as pJ-equivalent proxy values and are not meant to report physical energy consumption.*

We will:

1. Load and frame the event-based NMNIST dataset.
2. Train the same `fc -> N2D2 -> fc` classifier used in the Norse tutorial.
3. Count hidden-layer spikes on the test set and estimate a dynamic energy proxy.

The training split is restricted to a small subset by default so the notebook stays quick while still following a real event-data workflow.

## 1. Imports and requirements

This notebook requires additional python packages. You can install the tutorial dependencies with:

```bash
pip install -e .[examples]
```

In [ ]:
from pathlib import Path

import tonic
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset

from esn.neurons.clr import N2D2

SEED = 7
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

## 2. Load a small NMNIST workflow

NMNIST is a compact event-based digit dataset. Each sample is an event stream from a moving ATIS sensor. Here we use `tonic` to denoise the events and bin them into a fixed number of frames.

The `DataLoader` returns tensors with shape `[batch, time, polarity, height, width]`. The model below will flatten the spatial and polarity dimensions at each time step before applying the stateful spiking layer.

The first run may download NMNIST. After that, `tonic.DiskCachedDataset` stores the framed tensors so later runs start much faster.

In [ ]:
PROJECT_ROOT = Path("..").resolve()
DATA_ROOT = PROJECT_ROOT / ".data"
CACHE_ROOT = PROJECT_ROOT / ".cache" / "energy_nmnist"

n_steps = 25
batch_size = 64
train_subset = 1024
test_subset = 256
num_workers = 0
use_disk_cache = True


def maybe_subset(dataset, n_samples):
    if n_samples is None:
        return dataset
    return Subset(dataset, range(min(n_samples, len(dataset))))


def make_nmnist_dataset(train: bool, n_samples=None):
    transform = tonic.transforms.Compose([
        tonic.transforms.Denoise(filter_time=10_000),
        tonic.transforms.ToFrame(
            sensor_size=tonic.datasets.NMNIST.sensor_size,
            n_time_bins=n_steps,
        ),
    ])

    dataset = tonic.datasets.NMNIST(
        save_to=str(DATA_ROOT),
        train=train,
        transform=transform,
    )
    dataset = maybe_subset(dataset, n_samples)

    if use_disk_cache:
        split = "train" if train else "test"
        subset_name = "full" if n_samples is None else str(n_samples)
        cache_path = CACHE_ROOT / f"{split}_{subset_name}_{n_steps}_steps"
        dataset = tonic.DiskCachedDataset(dataset, cache_path=str(cache_path))

    return dataset


def make_loader(dataset, train: bool):
    generator = torch.Generator().manual_seed(SEED) if train else None
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=train,
        generator=generator,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=train,
    )


train_dataset = make_nmnist_dataset(train=True, n_samples=train_subset)
test_dataset = make_nmnist_dataset(train=False, n_samples=test_subset)
train_loader = make_loader(train_dataset, train=True)
test_loader = make_loader(test_dataset, train=False)

frames, labels = next(iter(train_loader))
print(f"Train samples: {len(train_dataset):,}")
print(f"Test samples:  {len(test_dataset):,}")
print(f"Labels: shape={tuple(labels.shape)}, first={labels[:8].tolist()}")

## 3. Define an SNN with an energy-aware ESN neuron

The model below uses the same architecture as the Norse tutorial:

- A linear layer maps input spikes into a hidden current.
- `N2D2()` turns that current into hidden spikes.
- A final linear layer maps hidden spikes to class logits.

The layers are kept explicit so the final section can count the hidden spikes produced by the evolved neuron. The neuron is instantiated with its default parameters because the built-in energy metadata is only valid for the shipped/default neuron configuration. The logits are averaged across time before the loss is computed.

In [ ]:
class EnergyESNClassifier(nn.Module):
    def __init__(
        self,
        input_features: int,
        hidden_features: int = 128,
        n_classes: int = 10,
        binarize_frames: bool = True,
    ):
        super().__init__()
        self.binarize_frames = binarize_frames
        self.input_layer = nn.Linear(input_features, hidden_features, bias=False)
        self.neuron = N2D2()
        self.readout = nn.Linear(hidden_features, n_classes)

    def forward(self, frames: torch.Tensor, return_hidden_spikes: bool = False):
        # frames: [batch, time, polarity, height, width]
        spikes = frames.float().flatten(start_dim=2)
        if self.binarize_frames:
            spikes = (spikes > 0).to(spikes.dtype)

        state = None
        logits_over_time = []
        hidden_spikes_over_time = []

        for x_t in spikes.transpose(0, 1):
            current_t = self.input_layer(x_t)
            hidden_spikes_t, state = self.neuron(current_t, state)
            logits_t = self.readout(hidden_spikes_t)
            logits_over_time.append(logits_t)
            hidden_spikes_over_time.append(hidden_spikes_t)

        logits = torch.stack(logits_over_time).mean(dim=0)
        if return_hidden_spikes:
            return logits, torch.stack(hidden_spikes_over_time)
        return logits


input_features = int(torch.tensor(tonic.datasets.NMNIST.sensor_size).prod().item())
model = EnergyESNClassifier(input_features=input_features).to(device)
print(model)

## 4. Train and evaluate

This is ordinary PyTorch training. The only SNN-specific detail is that the model consumes framed event data as a time sequence and internally carries neuron state across time.

In [ ]:
def run_epoch(model, loader, optimizer=None, max_batches=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    loss_fn = nn.CrossEntropyLoss()

    for batch_idx, (frames, labels) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        frames = frames.to(device)
        labels = labels.to(device)

        if training:
            optimizer.zero_grad(set_to_none=True)

        logits = model(frames)
        loss = loss_fn(logits, labels)

        if training:
            loss.backward()
            optimizer.step()

        batch_size = labels.numel()
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += batch_size

    return total_loss / total_samples, total_correct / total_samples


n_epochs = 3
learning_rate = 1e-3
max_train_batches = None
max_eval_batches = None

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(1, n_epochs + 1):
    train_loss, train_acc = run_epoch(model, train_loader, optimizer, max_batches=max_train_batches)
    test_loss, test_acc = run_epoch(model, test_loader, max_batches=max_eval_batches)
    print(
        f"epoch {epoch:02d} | "
        f"train loss {train_loss:.3f}, acc {train_acc:.3f} | "
        f"test loss {test_loss:.3f}, acc {test_acc:.3f}"
    )

## 5. Inspect predictions

The trained network returns one logit vector per sample. Higher logits indicate the predicted class.

In [ ]:
model.eval()
with torch.no_grad():
    frames, labels = next(iter(test_loader))
    logits = model(frames.to(device)).cpu()
    predictions = logits.argmax(dim=1)

print("labels:     ", labels[:16].tolist())
print("predictions:", predictions[:16].tolist())
print("frames shape:", tuple(frames.shape))
print("logits shape:", tuple(logits.shape))

## 6. Estimate test-set spike-energy proxy

Every evolved neuron stores FPGA-derived energy proxy metadata on the module. The values are reported in pJ-equivalent proxy units and are only valid for the default parameters of the shipped neuron instance.

Here we estimate the dynamic spike-energy proxy of the hidden `N2D2` layer on the test set. We first compute each hidden neuron's test-set spike rate, then pass those rates to the built-in `total_energy(...)` helper. If you tune thresholds, state dynamics, reset coefficients, or other neuron parameters, treat the example as a template and recompute the energy metadata for that modified neuron.

In [ ]:
def estimate_hidden_spike_energy(model, loader, max_batches=None):
    model.eval()

    hidden_spike_totals = None
    total_sample_steps = 0
    total_samples = 0
    total_correct = 0

    with torch.no_grad():
        for batch_idx, (frames, labels) in enumerate(loader):
            if max_batches is not None and batch_idx >= max_batches:
                break

            frames = frames.to(device)
            labels = labels.to(device)
            logits, hidden_spikes = model(frames, return_hidden_spikes=True)

            # hidden_spikes: [time, batch, hidden_features]
            batch_spike_totals = hidden_spikes.sum(dim=(0, 1)).cpu()
            if hidden_spike_totals is None:
                hidden_spike_totals = batch_spike_totals
            else:
                hidden_spike_totals += batch_spike_totals

            total_sample_steps += hidden_spikes.shape[0] * hidden_spikes.shape[1]
            total_samples += labels.numel()
            total_correct += (logits.argmax(dim=1) == labels).sum().item()

    spike_rates = hidden_spike_totals / total_sample_steps
    energy_per_sample_step = model.neuron.total_energy(spike_rates)
    total_energy = energy_per_sample_step * total_sample_steps

    return {
        "spike_rates": spike_rates,
        "n_samples": total_samples,
        "n_sample_steps": total_sample_steps,
        "accuracy": total_correct / total_samples,
        "energy_per_sample_step": energy_per_sample_step,
        "total_energy": total_energy,
    }


energy_unit = "pJ"
energy = estimate_hidden_spike_energy(model, test_loader, max_batches=max_eval_batches)
spike_rates = energy["spike_rates"]

print(f"Test samples: {energy['n_samples']:,}")
print(f"Test sample-steps: {energy['n_sample_steps']:,}")
print(f"Hidden neurons: {spike_rates.numel():,}")
print(f"Test accuracy: {energy['accuracy']:.3f}")
print(f"Mean hidden spike rate: {spike_rates.mean().item():.4f}")
print(f"Energy per sample step: {energy['energy_per_sample_step']:.2f} {energy_unit}")
print(f"Total energy: {energy['total_energy']:.2f} {energy_unit}")